# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [24]:

import os
import subprocess

if not os.path.exists("flyrank-ml-starter"):
    !git clone https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git

print("Current directory:", os.getcwd())
print("Repository exists:", os.path.exists("flyrank-ml-starter"))


Current directory: /content
Repository exists: True


In [25]:


import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

print("Imports successful.")

Imports successful.


In [26]:


import os
import glob
import subprocess

repo_path = "/content/flyrank-ml-starter"

if not os.path.exists(repo_path):
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/Nikita-Sudarshan/flyrank-ml-starter.git",
            repo_path
        ],
        check=True
    )

print("Repository exists:", os.path.exists(repo_path))

# Search the repository for the original dataset
matches = glob.glob(
    "/content/flyrank-ml-starter/**/content_refresh_anonymized.csv",
    recursive=True
)

print("Dataset files found:", matches)

if not matches:
    raise FileNotFoundError(
        "The original content_refresh_anonymized.csv is NOT in the GitHub repository."
    )

data_path = matches[0]

df = pd.read_csv(data_path)

print("Loaded from:", data_path)
print("Rows:", len(df))
print("Columns:", len(df.columns))

Repository exists: True
Dataset files found: ['/content/flyrank-ml-starter/data/raw/content_refresh_anonymized.csv']
Loaded from: /content/flyrank-ml-starter/data/raw/content_refresh_anonymized.csv
Rows: 30000
Columns: 44


In [27]:


required_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Dataset verified.")
print("Shape:", df.shape)

print("\nTrend direction:")
print(df["trend_direction"].value_counts())

print("\nTrend-related columns:")
print([
    col for col in df.columns
    if "trend" in col.lower()
])

Dataset verified.
Shape: (30000, 44)

Trend direction:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Trend-related columns:
['trend_direction', 'trend_pct']


In [28]:


# Target: 1 means the observed trend is "down"
y = df["trend_direction"].eq("down").astype(int)

# Client is used only for grouped validation, NOT as a model feature
groups = df["client_id"]

# Explicitly exclude the target and leakage-risk/context columns
excluded_columns = [
    "trend_direction",
    "trend_pct",
    "client_id"
]

# Keep all remaining columns as model features
X = df.drop(columns=excluded_columns)

print("Target:")
print("trend_direction → 1 = down, 0 = other")

print("\nTarget distribution:")
print(y.value_counts())

print("\nGroups:")
print("Unique clients:", groups.nunique())

print("\nFinal model features:")
print("Total features:", len(X.columns))
print(list(X.columns))

print("\nChecking excluded columns:")
leakage_columns_present = [
    col for col in excluded_columns
    if col in X.columns
]

if leakage_columns_present:
    print("FAIL — excluded columns found:", leakage_columns_present)
else:
    print("PASS — target and excluded leakage-risk columns are not model features")

Target:
trend_direction → 1 = down, 0 = other

Target distribution:
trend_direction
1    16262
0    13738
Name: count, dtype: int64

Groups:
Unique clients: 32

Final model features:
Total features: 41
['content_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']

Checking excluded columns:
PASS — target and excluded leakage-risk colu

In [29]:


from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

client_overlap = set(groups_train) & set(groups_test)

print("Client overlap:", len(client_overlap))

if len(client_overlap) == 0:
    print("PASS — no client appears in both train and test sets")
else:
    print("FAIL — client leakage detected")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
PASS — no client appears in both train and test sets


In [30]:


numeric_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

print("\nNumeric:")
print(numeric_features)

print("\nCategorical:")
print(categorical_features)

Numeric features: 29
Categorical features: 12

Numeric:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical:
['content_id', 'competition_level', 'content_type', 'main_intent', 'provider_used', 'model_used', 'age_tier', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']


In [31]:


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            numeric_features
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_features
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced"
            )
        )
    ]
)

print("Model pipeline created successfully.")

Model pipeline created successfully.


In [32]:


from sklearn.ensemble import RandomForestClassifier

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=50,
                random_state=42,
                n_jobs=-1,
                class_weight="balanced",
                max_depth=15,
                min_samples_leaf=5
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


In [33]:

y_prob = model.predict_proba(
    X_test
)[:, 1]

y_pred = (y_prob >= 0.5).astype(int)

print("Number of test predictions:", len(y_prob))
print("Minimum probability:", round(y_prob.min(), 4))
print("Maximum probability:", round(y_prob.max(), 4))
print("Mean probability:", round(y_prob.mean(), 4))

Number of test predictions: 6163
Minimum probability: 0.4794
Maximum probability: 0.5167
Mean probability: 0.5005


In [34]:


def precision_at_k(y_true, probabilities, k):
    order = np.argsort(probabilities)[::-1]
    top_k = order[:k]

    return y_true.iloc[top_k].mean()


k_values = [20, 50, 100, 200, 500, 1000]

print("Grouped-split model performance:\n")

for k in k_values:
    score = precision_at_k(
        y_test.reset_index(drop=True),
        y_prob,
        k
    )

    print(
        f"Precision@{k}: {score:.4f}"
    )

Grouped-split model performance:

Precision@20: 0.6000
Precision@50: 0.6200
Precision@100: 0.6300
Precision@200: 0.6800
Precision@500: 0.6300
Precision@1000: 0.5900


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [35]:


# The action queue ranks webpages by their predicted probability of decline.

# Pages with higher predicted decline probability are placed earlier in the review queue. Reason codes provide simple explanations based on observed page signals, such as high decline risk, stale content, or weaker recent performance.

# The queue is a prioritization tool for human review. It does not mean that a page should automatically be refreshed.

In [36]:


action_queue = X_test.copy()

# Add model score
action_queue["predicted_probability"] = y_prob

# Add observed outcome for evaluation
action_queue["observed_trend"] = df.iloc[test_idx][
    "trend_direction"
].values

# Generate human-readable reason codes
def generate_reason_codes(row):
    reasons = []

    if row["predicted_probability"] >= 0.80:
        reasons.append("DECLINE_RISK_HIGH")

    if (
        pd.notna(row["days_since_last_update"])
        and row["days_since_last_update"] > 180
    ):
        reasons.append("CONTENT_STALE")

    if (
        pd.notna(row["impressions_last_30d"])
        and pd.notna(row["impressions_prev_30d"])
        and row["impressions_last_30d"]
        < row["impressions_prev_30d"]
    ):
        reasons.append("RECENT_PERFORMANCE_WEAK")

    if not reasons:
        reasons.append("REVIEW_SIGNALS")

    return ", ".join(reasons)


action_queue["reason_codes"] = action_queue.apply(
    generate_reason_codes,
    axis=1
)

# Rank highest-risk pages first
action_queue = action_queue.sort_values(
    "predicted_probability",
    ascending=False
).reset_index(drop=True)

action_queue["priority_rank"] = (
    action_queue.index + 1
)

print("Action queue created.")
print("Total pages:", len(action_queue))

display(
    action_queue[
        [
            "priority_rank",
            "content_id",
            "predicted_probability",
            "reason_codes",
            "observed_trend"
        ]
    ].head(20)
)

Action queue created.
Total pages: 6163


,priority_rank,content_id,predicted_probability,reason_codes,observed_trend
0,1,content_714d862c87a1,0.516715,RECENT_PERFORMANCE_WEAK,stable
1,2,content_bcecc1a1c83c,0.516715,RECENT_PERFORMANCE_WEAK,down
2,3,content_7763d04703d0,0.516715,RECENT_PERFORMANCE_WEAK,stable
3,4,content_cdd4562de67a,0.516715,RECENT_PERFORMANCE_WEAK,down
4,5,content_b3f2ac089cc6,0.516715,RECENT_PERFORMANCE_WEAK,down
5,6,content_e83d674179f4,0.516715,RECENT_PERFORMANCE_WEAK,stable
6,7,content_5473f08b2acf,0.516715,RECENT_PERFORMANCE_WEAK,down
7,8,content_5ebe639dafce,0.516715,REVIEW_SIGNALS,stable
8,9,content_e069ebcf608e,0.516376,RECENT_PERFORMANCE_WEAK,down
9,10,content_6bada6a81afb,0.514625,RECENT_PERFORMANCE_WEAK,down


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [37]:


# This action queue is intended to support content teams in prioritizing webpages for human review.

# The model ranks pages by their predicted probability of belonging to the observed "down" trend category. Higher-ranked pages can be reviewed earlier for possible content refresh opportunities.

# The model is a decision-support tool, not an automatic refresh decision system. A high score does not prove that refreshing a page will improve performance.

# The results are based on historical observations and a grouped client split. They should not be interpreted as causal evidence or as guaranteed future outcomes.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [38]:


# A human should review every page before taking action based on the model's recommendation.

# The reviewer should check:

# - whether the content is still relevant to the intended search intent
# - whether the information is outdated
# - whether recent performance changes have another explanation
# - whether the page has strategic or business importance
# - whether the page type requires a different treatment
# - whether a refresh is actually appropriate

# ### No-go list

# The model should never automatically:

# - publish or edit content
# - delete a webpage
# - redirect a webpage
# - claim why search performance changed
# - make legal or compliance decisions
# - treat a high probability score as proof that a refresh will improve performance
# - replace human editorial approval

# The model output is limited to prioritization and decision-support.

In [39]:


review_queue = action_queue.copy()

review_queue["human_review_required"] = True
review_queue["no_automatic_action"] = True

print("Human review required for all recommendations:",
      review_queue["human_review_required"].all())

print("Automatic action disabled for all recommendations:",
      review_queue["no_automatic_action"].all())

print("\nTop 10 review queue:")
display(
    review_queue[
        [
            "priority_rank",
            "content_id",
            "predicted_probability",
            "reason_codes",
            "human_review_required",
            "no_automatic_action"
        ]
    ].head(10)
)

Human review required for all recommendations: True
Automatic action disabled for all recommendations: True

Top 10 review queue:


,priority_rank,content_id,predicted_probability,reason_codes,human_review_required,no_automatic_action
0,1,content_714d862c87a1,0.516715,RECENT_PERFORMANCE_WEAK,True,True
1,2,content_bcecc1a1c83c,0.516715,RECENT_PERFORMANCE_WEAK,True,True
2,3,content_7763d04703d0,0.516715,RECENT_PERFORMANCE_WEAK,True,True
3,4,content_cdd4562de67a,0.516715,RECENT_PERFORMANCE_WEAK,True,True
4,5,content_b3f2ac089cc6,0.516715,RECENT_PERFORMANCE_WEAK,True,True
5,6,content_e83d674179f4,0.516715,RECENT_PERFORMANCE_WEAK,True,True
6,7,content_5473f08b2acf,0.516715,RECENT_PERFORMANCE_WEAK,True,True
7,8,content_5ebe639dafce,0.516715,REVIEW_SIGNALS,True,True
8,9,content_e069ebcf608e,0.516376,RECENT_PERFORMANCE_WEAK,True,True
9,10,content_6bada6a81afb,0.514625,RECENT_PERFORMANCE_WEAK,True,True


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [40]:


# The recommendations may become stale if the data or the relationship between the observed signals and content decline changes over time.

# The model should be monitored for:

# - sustained decline in Precision@K on newly evaluated data
# - large changes in the distribution of predicted probabilities
# - changes in the distribution of important input features
# - changes in the mix of content types
# - changes in data collection or available fields
# - evidence that historical patterns no longer describe current content performance

# A retraining or investigation trigger should be based on measured degradation or meaningful data drift rather than an arbitrary calendar date.

# The current model results should therefore be treated as directional and decision-support evidence, not as a permanent performance guarantee.

In [41]:


monitoring = pd.Series(y_prob).describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)

print("Current prediction distribution:")
print(monitoring)

print("\nMonitoring summary:")
print("Prediction count:", len(y_prob))
print("Mean probability:", round(y_prob.mean(), 4))
print("Median probability:", round(np.median(y_prob), 4))
print("90th percentile:", round(np.percentile(y_prob, 90), 4))
print("95th percentile:", round(np.percentile(y_prob, 95), 4))
print("99th percentile:", round(np.percentile(y_prob, 99), 4))

Current prediction distribution:
count    6163.000000
mean        0.500504
std         0.006079
min         0.479360
50%         0.500208
75%         0.504580
90%         0.507565
95%         0.509694
99%         0.514625
max         0.516715
dtype: float64

Monitoring summary:
Prediction count: 6163
Mean probability: 0.5005
Median probability: 0.5002
90th percentile: 0.5076
95th percentile: 0.5097
99th percentile: 0.5146


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [42]:


# The ranked action queue is exported as a CSV so it can be reused by the research paper and deployed page.

# The export contains the priority rank, content identifier, predicted decline probability, reason codes, and observed trend.

# The exported queue is an analysis artifact and should be interpreted as decision-support output rather than an automated action list.

In [43]:


import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

paper_queue = review_queue[
    [
        "priority_rank",
        "content_id",
        "predicted_probability",
        "reason_codes",
        "observed_trend",
        "human_review_required",
        "no_automatic_action"
    ]
].copy()

output_path = os.path.join(
    output_dir,
    "content_action_queue.csv"
)

paper_queue.to_csv(
    output_path,
    index=False
)

print("Export created:")
print(output_path)

print("\nRows exported:", len(paper_queue))
print("Columns exported:", len(paper_queue.columns))

print("\nFirst 10 rows:")
display(paper_queue.head(10))

Export created:
work/outputs/content_action_queue.csv

Rows exported: 6163
Columns exported: 7

First 10 rows:


,priority_rank,content_id,predicted_probability,reason_codes,observed_trend,human_review_required,no_automatic_action
0,1,content_714d862c87a1,0.516715,RECENT_PERFORMANCE_WEAK,stable,True,True
1,2,content_bcecc1a1c83c,0.516715,RECENT_PERFORMANCE_WEAK,down,True,True
2,3,content_7763d04703d0,0.516715,RECENT_PERFORMANCE_WEAK,stable,True,True
3,4,content_cdd4562de67a,0.516715,RECENT_PERFORMANCE_WEAK,down,True,True
4,5,content_b3f2ac089cc6,0.516715,RECENT_PERFORMANCE_WEAK,down,True,True
5,6,content_e83d674179f4,0.516715,RECENT_PERFORMANCE_WEAK,stable,True,True
6,7,content_5473f08b2acf,0.516715,RECENT_PERFORMANCE_WEAK,down,True,True
7,8,content_5ebe639dafce,0.516715,REVIEW_SIGNALS,stable,True,True
8,9,content_e069ebcf608e,0.516376,RECENT_PERFORMANCE_WEAK,down,True,True
9,10,content_6bada6a81afb,0.514625,RECENT_PERFORMANCE_WEAK,down,True,True


In [44]:


import os
import pandas as pd

if not os.path.exists(output_path):
    raise FileNotFoundError(
        "The action queue export was not created."
    )

check = pd.read_csv(output_path)

print("PASS — export exists.")
print("Export shape:", check.shape)
print("Export path:", output_path)

print("\nExport columns:")
print(list(check.columns))

PASS — export exists.
Export shape: (6163, 7)
Export path: work/outputs/content_action_queue.csv

Export columns:
['priority_rank', 'content_id', 'predicted_probability', 'reason_codes', 'observed_trend', 'human_review_required', 'no_automatic_action']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [45]:

checks = {
    "Dataset is correct (30,000 rows)": len(df) == 30000,
    "Dataset has 44 columns": len(df.columns) == 44,
    "Target exists": "trend_direction" in df.columns,
    "Target excluded from model": "trend_direction" not in X.columns,
    "trend_pct excluded from model": "trend_pct" not in X.columns,
    "Client grouping has no overlap": len(set(groups_train) & set(groups_test)) == 0,
    "Grouped test set created": len(X_test) == 6163,
    "Predictions created": len(y_prob) == len(X_test),
    "Action queue created": len(action_queue) == len(X_test),
    "Paper export exists": os.path.exists(output_path),
}

print("ML-10 FINAL SELF-CHECK\n")

all_passed = True

for check, passed in checks.items():
    print(("PASS" if passed else "FAIL") + " — " + check)
    if not passed:
        all_passed = False

print("\n" + ("ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED"))

ML-10 FINAL SELF-CHECK

PASS — Dataset is correct (30,000 rows)
PASS — Dataset has 44 columns
PASS — Target exists
PASS — Target excluded from model
PASS — trend_pct excluded from model
PASS — Client grouping has no overlap
PASS — Grouped test set created
PASS — Predictions created
PASS — Action queue created
PASS — Paper export exists

ALL CHECKS PASSED
